# 01 · Perfilado del crudo ONSV

Fase CRISP-DM **Data Understanding**. Carga el XLSX real *tal como llega* y
documenta los problemas que la capa de limpieza debe resolver.

El objetivo NO es limpiar: es ver el dato sucio antes de decidir qué se limpia.

Preguntas que responde:

1. Donde esta la cabecera real del fichero?
2. Que columnas mezclan tipos y por qué no son escribibles en Parquet?
3. Que tanta informacion se pierde en cada limpieza?
4. Que reglas de dominio se pueden escribir a partir de la observacion?

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

from src.utils.configloader import expand_path, load_catalogs, load_settings, load_sources

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

SETTINGS = load_settings()
spec = load_sources()["onsv"].copy()
spec["path"] = expand_path(spec["path"])
print("Fuente:", spec["path"])
print("Existe:", Path(spec["path"]).exists())

## 1.1 · Donde esta la cabecera

El XLSX de ONSV tiene un preambulo institucional antes de los encabezados.
Leerlo con `pd.read_excel(path)` sin indicar `header` toma la primera fila
del preambulo como cabecera y todos los nombres quedan corruptos.

In [ ]:
naive = pd.read_excel(spec["path"], header=0, nrows=3)
correct = pd.read_excel(spec["path"], header=spec["header_row"], nrows=3)

print("Con header=0 (ingenuo), primeras columnas:")
print(list(naive.columns)[:6], "...")

print(f"Con header_row={spec['header_row']} (configurado):")
print(list(correct.columns))

## 1.2 · Nombres problematicos

Acentos, signos de interrogacion de apertura y espacios dobles. Cada uno
requiere una decision explicita de renombrado, no un `.strip()` generico.

In [ ]:
cols = list(correct.columns)
problemas = {
    'espacios dobles': [c for c in cols if '  ' in c],
    'signo apertura (¿)': [c for c in cols if '¿' in c],
    'acentos': [c for c in cols if any(ch in c for ch in 'ÁÉÍÓÚÑáéíóúñ')],
}
for k, v in problemas.items():
    print(f'{k}: {len(v)}')
    for c in v:
        print('    ', repr(c))

## 1.3 · Tipos mezclados (el problema que rompia Bronze)

Parquet exige **un tipo por columna**. Las columnas `object` que mezclan
tipos no son escribibles: PyArrow lanza
`Expected bytes, got a 'datetime.datetime' object`.

In [ ]:
import datetime as dt

raw = pd.read_excel(spec["path"], header=spec["header_row"])
print(f"Crudo: {raw.shape[0]:,} filas x {raw.shape[1]} columnas\n")

for col in raw.columns:
    s = raw[col]
    if s.dtype != object:
        continue
    tipos = set(type(v).__name__ for v in s.dropna())
    if len(tipos) > 1:
        print(f'{col!r} -> {sorted(tipos)}')
        for v in s.dropna().unique()[:3]:
            print('     ', repr(v))

### Comprobacion directa: se puede escribir en Parquet?

In [ ]:
import tempfile

tmp = Path(tempfile.gettempdir()) / "nb_bronze_test.parquet"
try:
    raw.to_parquet(tmp, engine="pyarrow", index=False)
    print("OK: el crudo se escribe directo en Parquet")
except Exception as exc:
    print("FALLA (como en produccion):", type(exc).__name__)
    print('  ', str(exc)[:140])

## 1.4 · La solucion aplicada: `_parquet_safe()`

Normaliza **por contenido**, no por nombre de columna, en tres pasos:
numerico -> fecha -> literal, aceptando la conversion solo si no destruye valores.

In [ ]:
from src.extract.register_raw import _parquet_safe

fixed = _parquet_safe(raw.copy())
print(f"Tras _parquet_safe: {fixed.shape[0]:,} filas x {fixed.shape[1]} columnas")
fixed.to_parquet(tmp, engine='pyarrow', index=False)
print("Se escribe en Parquet: si\n")

back = pd.read_parquet(tmp)
assert len(back) == len(raw), 'no se pierden filas'
print("Filas preservadas:", f"{len(back):,}")

cambiadas = [c for c in raw.columns if str(raw[c].dtype) != str(back[c].dtype)]
print('Columnas cuyo tipo cambio en Bronze:', cambiadas)

## 1.5 · Completitud y nulos por columna

Base para decidir imputaciones. Solo se imputa lo que tiene sentido imputar.

In [ ]:
nulos = pd.DataFrame({
    'nulos': raw.isna().sum(),
    'pct_nulos': (raw.isna().mean() * 100).round(2),
    'n_unicos': raw.nunique(dropna=True),
    'dtype': raw.dtypes.astype(str),
}).sort_values('pct_nulos', ascending=False)
nulos[nulos['pct_nulos'] > 0]

## 1.6 · Categoricas: que valores existen

Base empirica de `config/quality/catalogs/onsv.yaml`.

In [ ]:
for col in ["CLASE SINIESTRO", "CONDICIÓN CLIMÁTICA", "ZONA", "RED VIAL"]:
    if col not in raw.columns:
        continue
    print(f"\n=== {col} ({raw[col].nunique()} unicos) ===")
    print(raw[col].value_counts(dropna=False).head(12).to_string())

## 1.7 · El problema de los acentos

`CAÍDA DE PASAJERO` -> `CAIDA DE PASAJERO` tras `fold_accents`.
Si el catalogo mantiene el acento, la comparacion falla siempre.

In [ ]:
cats = load_catalogs("onsv")
print("Catalogo (forma plegada, la que produce Silver):")
for v in cats['clase']:
    print('   ', v)

print("\nValores en el crudo:")
cat_upper = {c.upper() for c in cats['clase']}
for v in sorted(raw["CLASE SINIESTRO"].dropna().unique()):
    marca = 'OK' if v.upper() in cat_upper else 'DIFIERE'
    print(f'    {v!r}  {marca}')

## 1.8 · Coordenadas: el Peru es negativo

Este detalle hizo fallar la dimension de validez en una version anterior:
un umbral `>= 0` aplicado a coordenadas de un pais del hemisferio sur.

In [ ]:
lat = pd.to_numeric(raw["COORDENADAS LATITUD"].astype(str)
                       .str.replace("°", "", regex=False), errors="coerce")
lon = pd.to_numeric(raw["COORDENADAS  LONGITUD"].astype(str)
                       .str.replace("°", "", regex=False), errors="coerce")

print("Latitud :", round(lat.min(), 4), "a", round(lat.max(), 4))
print("Longitud:", round(lon.min(), 4), "a", round(lon.max(), 4))
print(f"\nPct latitudes  negativas: {(lat < 0).mean() * 100:.1f}%")
print(f"Pct longitudes negativas: {(lon < 0).mean() * 100:.1f}%")
print("\n=> Un chequeo de validez >= 0 puntuaria 0% a ambas columnas.\n")
dentro = lat.between(-18.5, 0.2) & lon.between(-81.5, -68.5)
print(f"Dentro del territorio peruano: {dentro.mean() * 100:.1f}%")

## 1.9 · Fechas: DD/MM vs MM/DD

Pandas interpreta `01/05/2021` como 1 de mayo cuando el dato es 5 de enero.

In [ ]:
fechas_texto = raw["FECHA SINIESTRO"].dropna().astype(str)
print('Muestra:', fechas_texto.head(5).tolist(), '\n')
ingenua = pd.to_datetime(fechas_texto, errors="coerce")
correcta = pd.to_datetime(fechas_texto, format="%d/%m/%Y", errors="coerce")

print(f"NaT con parsing por defecto: {ingenua.isna().sum()}")
print(f"NaT con format=%d/%m/%Y : {correcta.isna().sum()}")
print(f"\nRango correcto: {correcta.min().date()} a {correcta.max().date()}")
print('Rango ingenuo :', ingenua.min(), 'a', ingenua.max())
print("\n=> Configurar date_columns en settings.yaml NO es opcional.")

## 1.10 · Duplicados en la clave primaria

In [ ]:
clave = "CÓDIGO SINIESTRO"
print("Filas        :", f"{len(raw):,}")
print("Codigos unicos:", f"{raw[clave].nunique():,}")
print("Duplicados   :", int(raw[clave].duplicated().sum()))
print("Nulos en la PK:", int(raw[clave].isna().sum()))

dup = raw[raw[clave].duplicated(keep=False)].sort_values(clave)
if len(dup):
    print('\nFilas implicadas:', len(dup))
    print(dup[[clave, 'FECHA SINIESTRO', 'CLASE SINIESTRO']].head(10).to_string(index=False))
else:
    print('\nNo hay duplicados: la PK codigo es unica de origen.')

## 1.11 · Perfil del target

`fallecidos` es el target de `model_ready`. Su distribucion determina que modelo
tendra sentido (conteo desbalanceado -> Poisson/Tweedie, no regresion lineal).

In [ ]:
f = pd.to_numeric(raw["CANTIDAD DE FALLECIDOS"], errors="coerce").fillna(0)
print(f.describe().to_string())
print("\nDistribucion de fallecidos por siniestro:")
print(f.astype(int).value_counts().sort_index().head(10).to_string())

## 1.12 · Variables categoricas frente al target

In [ ]:
f = pd.to_numeric(raw["CANTIDAD DE FALLECIDOS"], errors="coerce").fillna(0)
for col in ["CLASE SINIESTRO", "ZONA", "CONDICIÓN CLIMÁTICA"]:
    if col not in raw.columns:
        continue
    g = raw.assign(_f=f).groupby(col, dropna=False)["_f"].agg(["count", "mean"])
    g['pct'] = (g['count'] / len(raw) * 100).round(2)
    g['media_fallecidos'] = g['mean'].round(3)
    print(f"\n--- {col} ---")
    print(g[['pct', 'media_fallecidos']].head(10).to_string())
print(f"\nMedia global: {f.mean():.3f} fallecidos por siniestro")

## 1.14 · Visualización: distribución de las variables

Los números de las secciones anteriores en imagen. Aquí se ve la **firma de la
censurea de ONSV**: un pico en la última edad de `fallecidos`
(hasta 33 muertos en un mismo siniestro), que es donde se concentra el riesgo.
El DQS **no** detectaría ese valor: es un dato formalmente válido (positivo,
en rango según el contrato) ymuy extremo. Por eso los outliers se **reportan** y no
se borran: son información real sobre la cola de la distribución.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 110, 'savefig.bbox': 'tight'})
PALETTE = ['#4f46e5', '#059669', '#d97706', '#dc2626', '#0891b2']

# Mostrar en línea y capturar cualquier figura como imagen embebida.
from IPython.display import Image as _IPImage, display
import base64 as _b64
from pathlib import Path as _Path

def _show(png_path):
    """Muestra un PNG en línea dentro del notebook."""
    p = _Path(png_path)
    if not p.exists():
        print('Falta', p, '- ejecuta python scripts/graficos_etl.py')
        return
    display(_IPImage(data=_b64.b64decode(_b64.b64encode(p.read_bytes()))))
# En este notebook el dataframe disponible es el CRUDO (`raw`), y varias
# columnas llegan como `object` (tipos mezclados en el XLSX).
num = [c for c in ['CANTIDAD DE FALLECIDOS', 'CANTIDAD DE LESIONADOS',
                   'CANTIDAD DE VEHICULOS DAÑADOS', 'COORDENADAS LATITUD',
                   'COORDENADAS  LONGITUD'] if c in raw.columns]
num_df = raw[num].apply(pd.to_numeric, errors='coerce')
print('Variables numéricas del crudo:', list(num_df.columns))
print('Dtypes tras coerción:', num_df.dtypes.to_dict())
axes = num_df.hist(bins=30, figsize=(14, 7), color=PALETTE[0], edgecolor='white')
plt.tight_layout()
plt.show()

In [ ]:
skew = num_df.skew().round(2).sort_values(ascending=False)
print('Asimetría (skewness) por variable:')
print(skew.to_string())
print('\nInterpretación: |skew| > 1.5 = cola larga; > 3 = la media es poco representativa.')
plt.figure(figsize=(8, 4))
ax_sk = plt.gca()
ax_sk.barh(range(len(skew)), skew.values, color=PALETTE[0])
ax_sk.set_yticks(range(len(skew)))
ax_sk.set_yticklabels(skew.index, fontsize=9)
ax_sk.invert_yaxis()
plt.axvline(0, color='black', lw=1)
plt.axvline(1.5, color='#dc2626', ls='--', lw=1, label='umbral 1.5')
plt.axvline(-1.5, color='#dc2626', ls='--', lw=1)
plt.title('Asimetría de las variables numéricas')
plt.xlabel('skewness')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
f = pd.to_numeric(raw['CANTIDAD DE FALLECIDOS'], errors='coerce').dropna()
q1, q3 = f.quantile([0.25, 0.75])
iqr = q3 - q1
print(f'Descriptor:\n{f.describe().round(2).to_string()}')
print(f'\nIQR = {iqr:.1f} | umbral superior = {q3 + 1.5*iqr:.1f} fatalities')
n_out = int((f > q3 + 1.5*iqr).sum())
print(f'Outliers por IQR: {n_out} ({n_out/len(f)*100:.2f}%)')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(f, bins=25, color=PALETTE[0], edgecolor='white')
ax[0].axvline(q3 + 1.5*iqr, color='#dc2626', ls='--', label='umbral IQR')
ax[0].set_title('Distribución de fallecidos por siniestro')
ax[0].set_xlabel('fallecidos')
ax[0].legend()
sns.boxplot(x=f, ax=ax[1], color=PALETTE[0])
ax[1].set_title('Boxplot: la cola derecha concentra el riesgo')
plt.tight_layout()
plt.show()

## 1.15 · Correlación variables-objetivo

Qué variables se mueven con `fallecidos`. Solo numéricas: las categóricas
requieren codificación y se tratan aparte con boxplots.

In [ ]:
vars_num = [c for c in ['CANTIDAD DE FALLECIDOS', 'CANTIDAD DE LESIONADOS',
                          'CANTIDAD DE VEHICULOS DAÑADOS', 'COORDENADAS LATITUD',
                          'COORDENADAS  LONGITUD'] if c in raw.columns]
_num = raw[vars_num].apply(pd.to_numeric, errors='coerce')
corr = (_num.corr(numeric_only=True)['CANTIDAD DE FALLECIDOS']
        .drop('CANTIDAD DE FALLECIDOS').sort_values(key=abs, ascending=False))
print('Correlación de Pearson con fatalities:')
print(corr.round(3).to_string())
print('\nInterpretación: |r| < 0.2 = relación débil. Buscar alta correlación aquí sería una '
      'trampa: fatalities depende sobre todo del TIPO de siniestro, no de la geografía.')
plt.figure(figsize=(8, 4.5))
sns.barplot(x=corr.values, y=corr.index, palette='coolwarm')
plt.axvline(0, color='black', lw=1)
plt.title('Correlación con fatalities (Pearson)')
plt.tight_layout()
plt.show()

In [ ]:
print('Fallecidos medios por clase de siniestro:')
_f = pd.to_numeric(raw['CANTIDAD DE FALLECIDOS'], errors='coerce').fillna(0)
g = (raw.assign(_f=_f)
     .groupby('CLASE SINIESTRO', dropna=False)['_f']
     .agg(['count', 'mean', 'sum'])
     .sort_values('mean', ascending=False))
g['media_fallecidos'] = g['mean'].round(2)
print(g[['count', 'media_fallecidos', 'sum']].head(12).to_string())
top = g.head(12)
plt.figure(figsize=(9, 4.5))
ax_cls = plt.gca()
ax_cls.barh(range(len(top)), top['media_fallecidos'].values, color=PALETTE[2])
ax_cls.set_yticks(range(len(top)))
ax_cls.set_yticklabels(top.index, fontsize=9)
ax_cls.invert_yaxis()
plt.axvline(_f.mean(), color='#dc2626', ls='--',
            label=f'media global = {_f.mean():.2f}')
plt.xlabel('fallecidos medios por siniestro')
plt.title('Fallecidos medios por clase: aquí SÍ está la señal')
plt.legend()
plt.tight_layout()
plt.show()

## 1.13 · Conclusiones

| Hallazgo | Decision de diseno |
|---|---|
| Preambulo de 4 filas | `header_row: 4` en la config de la fuente |
| Nombres con acentos, `¿?` y espacios dobles | mapa `rename` canonico |
| Fechas DD/MM ambiguas | `date_columns` con formatos explicitos |
| Coordenadas con simbolo grado | `numeric_strip_chars` |
| Columnas `object` con tipos mezclados | `_parquet_safe()` en Bronze |
| Acentos en categorias | catalogos en forma plegada + comparacion `_norm()` |
| Coordenadas negativas (Peru) | `validity` usa el rango del contrato |

Todos estos puntos estan cubiertos por `tests/unit/test_regressions.py`.